# TUJI1: KNN centralizado, distribuido y por clúster

Mantiene el test oficial y obtiene validación solo desde train, agrupando las adquisiciones por posición tridimensional.


## 1. Objetivo y protocolo experimental

Se comparan cuatro modalidades: KNN centralizado global, KNN distribuido
global, KNN centralizado por clúster predicho y KNN distribuido por clúster
predicho. Los hiperparámetros y el número de clústeres se eligen únicamente
con validación. El test se evalúa después de fijar cada ganador.


## 2. Importaciones y configuración del entorno


In [22]:
from pathlib import Path
from dataclasses import asdict, dataclass
from itertools import product
from typing import Dict, List, Mapping, Optional, Sequence, Tuple
import copy
import json
import math
import random
import re
import warnings

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.cluster import KMeans
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.neighbors import (
    KNeighborsClassifier,
    KNeighborsRegressor,
    NearestNeighbors,
)
from sklearn.preprocessing import LabelEncoder, StandardScaler

try:
    from IPython.display import display
except ImportError:
    display = print


# Si Jupyter se inicia fuera de la carpeta del paquete, escribe aquí su ruta.
# Ejemplo: PROJECT_ROOT = Path(r"C:/TFM/notebooks_autocontenidos_sin_core")
PROJECT_ROOT = None

cwd = Path.cwd().resolve()
root_candidates = [cwd, cwd.parent, cwd.parent.parent]
if PROJECT_ROOT is not None:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
else:
    ROOT = next(
        (
            candidate
            for candidate in root_candidates
            if sum((candidate / name).is_dir() for name in ["TUT", "TUJI1", "UJIIndoor", "SOD"])
            >= 2
        ),
        cwd,
    )

SEED = 42
TARGET_COLUMNS = ["TARGET_X_M", "TARGET_Y_M"]

print("Raíz utilizada:", ROOT)


Raíz utilizada: /home/coder/Indoor/Notebooks


### 2.1. Carga y preprocesado

Estas funciones están dentro del notebook. Detectan las columnas
RSSI, cargan las particiones creadas por el notebook `00` y aplican
una transformación ajustada exclusivamente con `train`.


In [23]:
def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def natural_key(text: str) -> List[object]:
    return [int(piece) if piece.isdigit() else piece for piece in re.split(r"(\d+)", text)]


def detect_rssi_columns(df: pd.DataFrame) -> List[str]:
    cols = [c for c in df.columns if re.fullmatch(r"(?:WAP|MAC)\d+", str(c).upper())]
    cols = sorted(cols, key=natural_key)
    if not cols:
        raise ValueError("No se detectaron columnas RSSI WAPnnn o MACnnn.")
    return cols


class RSSIPreprocessor:
    """Imputa ausencias, estandariza RSSI con train y anade mascara de deteccion."""

    def __init__(self, missing_value: float = 100.0, fill_value: float = -110.0, use_mask: bool = True):
        self.missing_value = float(missing_value)
        self.fill_value = float(fill_value)
        self.use_mask = bool(use_mask)
        self.scaler = StandardScaler()
        self.columns: List[str] = []

    def _clean(self, df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        raw = df[self.columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
        observed = np.isfinite(raw) & (raw != self.missing_value)
        clean = np.where(observed, raw, self.fill_value).astype(np.float32)
        return clean, observed.astype(np.float32)

    def fit(self, df: pd.DataFrame, columns: Sequence[str]) -> "RSSIPreprocessor":
        self.columns = list(columns)
        clean, _ = self._clean(df)
        self.scaler.fit(clean)
        return self

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        clean, mask = self._clean(df)
        scaled = self.scaler.transform(clean).astype(np.float32)
        if self.use_mask:
            return np.concatenate([scaled, mask], axis=1).astype(np.float32)
        return scaled

    def fit_transform(self, df: pd.DataFrame, columns: Sequence[str]) -> np.ndarray:
        return self.fit(df, columns).transform(df)


#### Lectura de particiones y rutas


In [24]:
def read_base_splits(output_dir: Path, prefix: str) -> Dict[str, pd.DataFrame]:
    output_dir = Path(output_dir)
    return {
        split: pd.read_csv(output_dir / f"{prefix}_{split}.csv")
        for split in ["train", "val", "test"]
    }


def load_prepared_bundle(prepared_dir: Path, prefix: str) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame]:
    prepared_dir = Path(prepared_dir)
    splits = read_base_splits(prepared_dir, prefix)
    routes = pd.read_csv(prepared_dir / f"{prefix}_routes.csv")
    expected = {"ROW_ID", "SPLIT", "N_CLUSTERS", "CLUSTER", "CLUSTER_ORACLE"}
    if not expected.issubset(routes.columns):
        raise ValueError(f"El fichero de rutas no contiene {sorted(expected)}")
    return splits, routes


def route_frame(frame: pd.DataFrame, routes: pd.DataFrame, split: str, n_clusters: int) -> pd.DataFrame:
    selected = routes[
        (routes["SPLIT"].astype(str) == split)
        & (pd.to_numeric(routes["N_CLUSTERS"]) == int(n_clusters))
    ].copy()
    out = frame.merge(selected, on="ROW_ID", how="left", validate="one_to_one")
    if out["CLUSTER"].isna().any():
        raise ValueError(f"Faltan rutas para {split}, K={n_clusters}.")
    out["CLUSTER"] = out["CLUSTER"].astype(int)
    out["CLUSTER_ORACLE"] = out["CLUSTER_ORACLE"].astype(int)
    return out


### 2.2. Métricas y enrutamiento por clúster

El error de coordenadas es la distancia radial 2D en metros. Para
los modelos por zona, `CLUSTER` es la salida del router RSSI;
`CLUSTER_ORACLE` solo se utiliza como diagnóstico.


In [25]:
def coordinate_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    truth = np.asarray(y_true, dtype=float)
    pred = np.asarray(y_pred, dtype=float)
    if truth.shape != pred.shape or truth.ndim != 2 or truth.shape[1] != 2:
        raise ValueError(f"Se esperaban matrices (n,2); recibidas {truth.shape} y {pred.shape}.")
    distances = np.linalg.norm(pred - truth, axis=1)
    return {
        "rmse_2d_m": float(np.sqrt(np.mean(distances ** 2))),
        "mean_error_m": float(np.mean(distances)),
        "median_error_m": float(np.median(distances)),
        "p75_error_m": float(np.quantile(distances, 0.75)),
        "p95_error_m": float(np.quantile(distances, 0.95)),
        "n": int(len(distances)),
    }


def _metric_row(
    model: str,
    modality: str,
    split: str,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    n_clusters: Optional[int] = None,
    extra: Optional[Mapping[str, object]] = None,
) -> Dict[str, object]:
    row: Dict[str, object] = {
        "model": model,
        "modality": modality,
        "split": split,
        "n_clusters": n_clusters,
        **coordinate_metrics(y_true, y_pred),
    }
    if extra:
        row.update(dict(extra))
    return row


def _predict_by_zone(
    models: Mapping[int, object],
    fallback: object,
    x: np.ndarray,
    zone_labels: np.ndarray,
) -> Tuple[np.ndarray, float]:
    pred = np.empty((len(x), 2), dtype=float)
    covered = np.zeros(len(x), dtype=bool)
    zones = np.asarray(zone_labels, dtype=int)
    for zone in np.unique(zones):
        mask = zones == zone
        model = models.get(int(zone), fallback)
        pred[mask] = model.predict(x[mask])
        covered[mask] = int(zone) in models
    return pred, float(np.mean(covered))


def compact_results(result: pd.DataFrame, split: str = "test") -> pd.DataFrame:
    columns = [
        "model",
        "modality",
        "n_clusters",
        "rmse_2d_m",
        "mean_error_m",
        "median_error_m",
        "p75_error_m",
        "p95_error_m",
        "coverage",
        "gate_accuracy_diagnostic",
    ]
    available = [c for c in columns if c in result.columns]
    return result[result["split"] == split][available].sort_values("rmse_2d_m").reset_index(drop=True)


## 3. Ficheros preparados y configuración del experimento


In [26]:
DATASET = 'TUJI1'
PREPARED_DIR = ROOT / "prepared" / 'TUJI1'
PREFIX = 'tuji1_official'
RESULTS_DIR = ROOT / "results" / 'TUJI1'

COORD_NEIGHBORS = (1, 3, 5, 7, 11, 15, 21, 31)
FLOOR_NEIGHBORS = ()
DISTANCE_METRICS = ("manhattan", "euclidean")
WEIGHT_OPTIONS = ("uniform", "distance")
CLUSTER_VALUES = [2, 3, 4, 5, 6, 7, 8, 9, 10]
FLOOR_TASK = False
REQUIRE_KNOWN_TEST_CLIENTS = False

COORD_RESULTS_PATH = RESULTS_DIR / 'tuji1_official_knn_coordinates_corrected.csv'
COORD_TUNING_PATH = COORD_RESULTS_PATH.with_name(COORD_RESULTS_PATH.stem + "_tuning.csv")
COORD_CLUSTER_TUNING_PATH = COORD_RESULTS_PATH.with_name(
    COORD_RESULTS_PATH.stem + "_cluster_tuning.csv"
)
COORD_SELECTED_PATH = COORD_RESULTS_PATH.with_name(
    COORD_RESULTS_PATH.stem + "_selected_hyperparameters.json"
)

if FLOOR_TASK:
    FLOOR_RESULTS_PATH = RESULTS_DIR / ''
    FLOOR_TUNING_PATH = FLOOR_RESULTS_PATH.with_name(FLOOR_RESULTS_PATH.stem + "_tuning.csv")
    FLOOR_CLUSTER_TUNING_PATH = FLOOR_RESULTS_PATH.with_name(
        FLOOR_RESULTS_PATH.stem + "_cluster_tuning.csv"
    )
    FLOOR_SELECTED_PATH = FLOOR_RESULTS_PATH.with_name(
        FLOOR_RESULTS_PATH.stem + "_selected_hyperparameters.json"
    )


In [27]:
required_inputs = [
    PREPARED_DIR / f"{PREFIX}_train.csv",
    PREPARED_DIR / f"{PREFIX}_val.csv",
    PREPARED_DIR / f"{PREFIX}_test.csv",
    PREPARED_DIR / f"{PREFIX}_routes.csv",
]
missing_inputs = [path for path in required_inputs if not path.exists()]

if missing_inputs:
    formatted = "\n- ".join(str(path) for path in missing_inputs)
    raise FileNotFoundError(
        "Faltan las particiones preparadas:\n- " + formatted
        + "\nEjecuta primero el notebook 00 del dataset incluido en este paquete."
    )

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Raíz del proyecto:", ROOT)
print("Datos preparados:", PREPARED_DIR)
print("Resultados:", RESULTS_DIR)


Raíz del proyecto: /home/coder/Indoor/Notebooks
Datos preparados: /home/coder/Indoor/Notebooks/prepared/TUJI1
Resultados: /home/coder/Indoor/Notebooks/results/TUJI1


## 4. Carga de datos

Las particiones y las rutas RSSI→clúster proceden del notebook `00` del
dataset. No se reutilizan etiquetas de clúster calculadas con las
coordenadas reales de validación o test.


In [28]:
splits, routes = load_prepared_bundle(PREPARED_DIR, PREFIX)

position_columns = ["TARGET_X_M", "TARGET_Y_M"]
if "FLOOR_LABEL" in splits["train"].columns:
    position_columns.append("FLOOR_LABEL")

partition_summary = []
for split_name, frame in splits.items():
    partition_summary.append(
        {
            "split": split_name,
            "rows": len(frame),
            "positions": frame[position_columns].drop_duplicates().shape[0],
            "clients": frame["CLIENT_ID"].astype(str).nunique(),
            "floors": (
                frame["FLOOR_LABEL"].nunique()
                if "FLOOR_LABEL" in frame.columns
                else np.nan
            ),
        }
    )

display(pd.DataFrame(partition_summary).set_index("split"))
print("Valores de K disponibles:", sorted(routes["N_CLUSTERS"].astype(int).unique()))


,rows,positions,clients,floors
split,,,,
train,5740,1148,5,NaN
val,1012,203,5,NaN
test,2147,431,5,NaN


Valores de K disponibles: [np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]


## 5. Auditoría de las particiones


In [29]:
def _row_ids(frame):
    return set(frame["ROW_ID"].astype(str))


def _positions(frame):
    return set(map(tuple, frame[position_columns].astype(str).to_numpy()))


audit_rows = []
for left, right in [("train", "val"), ("train", "test"), ("val", "test")]:
    audit_rows.append(
        {
            "partitions": f"{left}-{right}",
            "shared_row_ids": len(_row_ids(splits[left]) & _row_ids(splits[right])),
            "shared_positions": len(_positions(splits[left]) & _positions(splits[right])),
        }
    )

display(pd.DataFrame(audit_rows).set_index("partitions"))

train_clients = set(splits["train"]["CLIENT_ID"].astype(str))
val_clients = set(splits["val"]["CLIENT_ID"].astype(str))
test_clients = set(splits["test"]["CLIENT_ID"].astype(str))
unseen_vs_train = sorted(test_clients - train_clients)
unseen_vs_train_val = sorted(test_clients - (train_clients | val_clients))

print("Clientes de test ausentes en train:", unseen_vs_train)
print("Clientes de test ausentes en train/validación:", unseen_vs_train_val)

assert len(_row_ids(splits["train"]) & _row_ids(splits["val"])) == 0
assert len(_positions(splits["train"]) & _positions(splits["val"])) == 0
if REQUIRE_KNOWN_TEST_CLIENTS:
    assert unseen_vs_train_val == [], (
        "UJIIndoor contiene dispositivos de test no presentes en train/validación: "
        f"{unseen_vs_train_val}. Vuelve a ejecutar el notebook 00 corregido."
    )

print("Auditoría superada.")


,shared_row_ids,shared_positions
partitions,,
train-val,0,0
train-test,0,0
val-test,0,0


Clientes de test ausentes en train: []
Clientes de test ausentes en train/validación: []
Auditoría superada.


## 6. Variables disponibles


In [30]:
rssi_columns = detect_rssi_columns(splits["train"])

variable_summary = pd.DataFrame(
    {
        "group": ["RSSI", "target", "client", "routing"],
        "columns": [
            len(rssi_columns),
            len([column for column in ["TARGET_X_M", "TARGET_Y_M", "FLOOR_LABEL"] if column in splits["train"]]),
            1,
            len([column for column in ["ROW_ID", "SPLIT"] if column in splits["train"]]),
        ],
    }
)
display(variable_summary.set_index("group"))
print("Primeras columnas RSSI:", rssi_columns[:10])
print("Rango RSSI en train:", float(splits["train"][rssi_columns].min().min()), "a", float(splits["train"][rssi_columns].max().max()))


,columns
group,
RSSI,310
target,2
client,1
routing,1


Primeras columnas RSSI: ['WAP001', 'WAP002', 'WAP003', 'WAP004', 'WAP005', 'WAP006', 'WAP007', 'WAP008', 'WAP009', 'WAP010']
Rango RSSI en train: -101.0 a 100.0


## 7. Variables objetivo y métricas

La regresión usa `TARGET_X_M` y `TARGET_Y_M`. La selección se realiza con
RMSE radial 2D en metros; también se guardan error medio, mediana, P75 y
P95. Cuando existe `FLOOR_LABEL`, la clasificación de planta se selecciona
mediante accuracy.


## 8. Preprocesado RSSI


In [31]:
# Esta celda solo documenta las dimensiones. El experimento vuelve a ajustar
# internamente un preprocesador idéntico usando exclusivamente train.
preprocessor_preview = RSSIPreprocessor(use_mask=True).fit(splits["train"], rssi_columns)
feature_shapes = {
    split_name: preprocessor_preview.transform(frame).shape
    for split_name, frame in splits.items()
}
display(
    pd.DataFrame(
        [
            {"split": name, "samples": shape[0], "features_after_mask": shape[1]}
            for name, shape in feature_shapes.items()
        ]
    ).set_index("split")
)
print("El escalador y la imputación se ajustan únicamente con train.")


,samples,features_after_mask
split,,
train,5740,620
val,1012,620
test,2147,620


El escalador y la imputación se ajustan únicamente con train.


## 9. Modelos KNN

El modelo distribuido no promedia una predicción independiente por
cliente. Cada cliente recupera sus candidatos locales y el servidor
fusiona todos los candidatos antes de seleccionar los vecinos globales.
Así reconstruye el mismo conjunto de vecinos que el KNN centralizado,
salvo posibles empates exactos de distancia.

En las modalidades por clúster, validación y test se asignan mediante el
router entrenado solo con RSSI. `CLUSTER_ORACLE` se usa exclusivamente como
diagnóstico y nunca para elegir la predicción final.


### 9.1. Implementación autocontenida de KNN

Aquí se definen el modelo central, su estrategia distribuida o
federada y el procedimiento completo de selección con validación.
No se importa código propio desde ningún fichero `.py`.


#### Modelo KNN distribuido


In [32]:
class DistributedKNNRegressor(BaseEstimator, RegressorMixin):
    """KNN horizontal: cada cliente devuelve top-k y el servidor fusiona top-k global."""

    def __init__(self, n_neighbors: int = 5, metric: str = "manhattan", weights: str = "distance"):
        self.n_neighbors = int(n_neighbors)
        self.metric = metric
        self.weights = weights
        self.clients_: List[Tuple[NearestNeighbors, np.ndarray]] = []

    def fit(self, x: np.ndarray, y: np.ndarray, client_ids: Sequence[object]):
        self.clients_ = []
        ids = np.asarray(client_ids).astype(str)
        for client in sorted(np.unique(ids)):
            mask = ids == client
            x_client = np.asarray(x)[mask]
            y_client = np.asarray(y)[mask]
            if not len(x_client):
                continue
            nn = NearestNeighbors(
                n_neighbors=min(self.n_neighbors, len(x_client)), metric=self.metric, n_jobs=-1
            ).fit(x_client)
            self.clients_.append((nn, y_client))
        if not self.clients_:
            raise ValueError("No hay clientes con datos para KNN distribuido.")
        return self

    def predict(self, x: np.ndarray) -> np.ndarray:
        queries = np.asarray(x)
        all_distances: List[np.ndarray] = []
        all_targets: List[np.ndarray] = []
        for nn, targets in self.clients_:
            distances, indices = nn.kneighbors(queries)
            all_distances.append(distances)
            all_targets.append(targets[indices])
        distances = np.concatenate(all_distances, axis=1)
        targets = np.concatenate(all_targets, axis=1)
        k = min(self.n_neighbors, distances.shape[1])
        chosen = np.argpartition(distances, kth=k - 1, axis=1)[:, :k]
        d = np.take_along_axis(distances, chosen, axis=1)
        y = np.take_along_axis(targets, chosen[:, :, None], axis=1)
        if self.weights == "uniform":
            return y.mean(axis=1)
        output = np.empty((len(queries), y.shape[2]), dtype=float)
        for i in range(len(queries)):
            zero = d[i] <= 1e-12
            if np.any(zero):
                output[i] = y[i, zero].mean(axis=0)
            else:
                w = 1.0 / d[i]
                output[i] = np.average(y[i], axis=0, weights=w)
        return output


def _fit_knn_pair(
    x: np.ndarray,
    y: np.ndarray,
    clients: np.ndarray,
    n_neighbors: int,
    metric: str,
    weights: str,
) -> Tuple[KNeighborsRegressor, DistributedKNNRegressor]:
    k_eff = max(1, min(int(n_neighbors), len(x)))
    central = KNeighborsRegressor(
        n_neighbors=k_eff, weights=weights, metric=metric, n_jobs=-1
    ).fit(x, y)
    distributed = DistributedKNNRegressor(k_eff, metric=metric, weights=weights).fit(
        x, y, clients
    )
    return central, distributed


#### Selección y evaluación KNN


In [33]:
def run_knn_experiment(
    prepared_dir: Path,
    prefix: str,
    neighbor_values: Sequence[int] = (3, 5, 7, 9),
    metrics: Sequence[str] = ("manhattan", "euclidean"),
    weights_values: Sequence[str] = ("uniform", "distance"),
    cluster_values: Optional[Sequence[int]] = None,
    output_csv: Optional[Path] = None,
) -> pd.DataFrame:
    seed_everything()
    splits, routes = load_prepared_bundle(prepared_dir, prefix)
    rssi = detect_rssi_columns(splits["train"])
    pre = RSSIPreprocessor(use_mask=True).fit(splits["train"], rssi)
    x = {name: pre.transform(frame) for name, frame in splits.items()}
    y = {name: frame[TARGET_COLUMNS].to_numpy(dtype=float) for name, frame in splits.items()}
    clients = splits["train"]["CLIENT_ID"].astype(str).to_numpy()

    tuning: List[Dict[str, object]] = []
    fitted: Dict[Tuple[int, str, str], Tuple[object, object]] = {}
    for k in neighbor_values:
        for metric in metrics:
            for weights in weights_values:
                central, dist = _fit_knn_pair(
                    x["train"], y["train"], clients, k, metric, weights
                )
                fitted[(int(k), metric, weights)] = (central, dist)
                for name, model in [("central", central), ("distributed", dist)]:
                    score = coordinate_metrics(y["val"], model.predict(x["val"]))
                    tuning.append(
                        {
                            "kind": name,
                            "n_neighbors": int(k),
                            "distance": metric,
                            "weights": weights,
                            **score,
                        }
                    )
    tuning_df = pd.DataFrame(tuning)
    best_central = tuning_df[tuning_df["kind"] == "central"].sort_values("rmse_2d_m").iloc[0]
    best_k = int(best_central["n_neighbors"])
    best_metric = str(best_central["distance"])
    best_weights = str(best_central["weights"])
    central_global, distributed_global = fitted[(best_k, best_metric, best_weights)]

    rows: List[Dict[str, object]] = []
    for split in ["val", "test"]:
        for modality, model in [
            ("centralized_global", central_global),
            ("federated_global_exact_distributed_knn", distributed_global),
        ]:
            rows.append(
                _metric_row(
                    "KNN",
                    modality,
                    split,
                    y[split],
                    model.predict(x[split]),
                    extra={
                        "n_neighbors": best_k,
                        "distance": best_metric,
                        "weights": best_weights,
                    },
                )
            )

    available_k = sorted(pd.to_numeric(routes["N_CLUSTERS"]).astype(int).unique())
    cluster_values = list(cluster_values) if cluster_values is not None else available_k
    cluster_values = [int(k) for k in cluster_values if int(k) in available_k]
    cluster_cache: Dict[int, Tuple[Dict[int, object], Dict[int, object]]] = {}
    cluster_val_rows: List[Dict[str, object]] = []
    for k_clusters in cluster_values:
        routed = {
            split: route_frame(splits[split], routes, split, k_clusters)
            for split in ["train", "val"]
        }
        central_zone: Dict[int, object] = {}
        distributed_zone: Dict[int, object] = {}
        for zone in sorted(routed["train"]["CLUSTER"].unique()):
            mask = routed["train"]["CLUSTER"].to_numpy() == zone
            if not np.any(mask):
                continue
            c_model, d_model = _fit_knn_pair(
                x["train"][mask], y["train"][mask], clients[mask],
                best_k, best_metric, best_weights
            )
            central_zone[int(zone)] = c_model
            distributed_zone[int(zone)] = d_model
        cluster_cache[k_clusters] = (central_zone, distributed_zone)
        labels = routed["val"]["CLUSTER"].to_numpy(dtype=int)
        cpred, ccov = _predict_by_zone(central_zone, central_global, x["val"], labels)
        dpred, dcov = _predict_by_zone(distributed_zone, distributed_global, x["val"], labels)
        gate_acc = float(
            np.mean(labels == routed["val"]["CLUSTER_ORACLE"].to_numpy())
        )
        for modality, pred, coverage in [
            ("centralized_by_predicted_cluster", cpred, ccov),
            ("federated_by_predicted_cluster", dpred, dcov),
        ]:
            cluster_val_rows.append(
                _metric_row(
                    "KNN",
                    modality,
                    "val",
                    y["val"],
                    pred,
                    n_clusters=k_clusters,
                    extra={
                        "coverage": coverage,
                        "gate_accuracy_diagnostic": gate_acc,
                        "n_neighbors": best_k,
                        "distance": best_metric,
                        "weights": best_weights,
                    },
                )
            )

    cluster_tuning_df = pd.DataFrame(cluster_val_rows)
    for modality in ["centralized_by_predicted_cluster", "federated_by_predicted_cluster"]:
        val_rows = cluster_tuning_df[cluster_tuning_df["modality"] == modality]
        if val_rows.empty:
            continue
        best_cluster_k = int(val_rows.sort_values("rmse_2d_m").iloc[0]["n_clusters"])
        rows.append(val_rows[val_rows["n_clusters"] == best_cluster_k].iloc[0].to_dict())
        test_routed = route_frame(splits["test"], routes, "test", best_cluster_k)
        test_labels = test_routed["CLUSTER"].to_numpy(dtype=int)
        central_zone, distributed_zone = cluster_cache[best_cluster_k]
        if modality == "centralized_by_predicted_cluster":
            test_pred, coverage = _predict_by_zone(
                central_zone, central_global, x["test"], test_labels
            )
        else:
            test_pred, coverage = _predict_by_zone(
                distributed_zone, distributed_global, x["test"], test_labels
            )
        rows.append(
            _metric_row(
                "KNN",
                modality,
                "test",
                y["test"],
                test_pred,
                n_clusters=best_cluster_k,
                extra={
                    "coverage": coverage,
                    "gate_accuracy_diagnostic": float(
                        np.mean(test_labels == test_routed["CLUSTER_ORACLE"].to_numpy())
                    ),
                    "n_neighbors": best_k,
                    "distance": best_metric,
                    "weights": best_weights,
                },
            )
        )
    keep = pd.DataFrame(rows).sort_values(["split", "modality"]).reset_index(drop=True)
    if output_csv is not None:
        path = Path(output_csv)
        path.parent.mkdir(parents=True, exist_ok=True)
        keep.to_csv(path, index=False)
        tuning_df.to_csv(path.with_name(path.stem + "_tuning.csv"), index=False)
        cluster_tuning_df.to_csv(
            path.with_name(path.stem + "_cluster_tuning.csv"), index=False
        )
        with path.with_name(path.stem + "_selected_hyperparameters.json").open(
            "w", encoding="utf-8"
        ) as handle:
            json.dump(
                {
                    "model": "KNN",
                    "task": "coordinates",
                    "selection_split": "validation",
                    "selection_metric": "rmse_2d_m",
                    "n_neighbors": best_k,
                    "distance": best_metric,
                    "weights": best_weights,
                    "cluster_protocol": "K is selected independently by modality on validation.",
                },
                handle,
                indent=2,
                ensure_ascii=False,
            )
    return keep


## 10. Espacio de hiperparámetros


In [34]:
coordinate_grid = pd.DataFrame(
    list(product(COORD_NEIGHBORS, DISTANCE_METRICS, WEIGHT_OPTIONS)),
    columns=["n_neighbors", "distance", "weights"],
)
print("Candidatos para coordenadas:", len(coordinate_grid))
display(coordinate_grid)

if FLOOR_TASK:
    floor_grid = pd.DataFrame(
        list(product(FLOOR_NEIGHBORS, DISTANCE_METRICS, WEIGHT_OPTIONS)),
        columns=["n_neighbors", "distance", "weights"],
    )
    print("Candidatos para planta:", len(floor_grid))
    display(floor_grid)

print("Valores de K espacial evaluados:", CLUSTER_VALUES)


Candidatos para coordenadas: 32


,n_neighbors,distance,weights
0,1,manhattan,uniform
1,1,manhattan,distance
2,1,euclidean,uniform
3,1,euclidean,distance
4,3,manhattan,uniform
5,3,manhattan,distance
6,3,euclidean,uniform
7,3,euclidean,distance
8,5,manhattan,uniform
9,5,manhattan,distance


Valores de K espacial evaluados: [2, 3, 4, 5, 6, 7, 8, 9, 10]


## 11. Entrenamiento y evaluación de coordenadas


In [35]:
coordinate_results = run_knn_experiment(
    prepared_dir=PREPARED_DIR,
    prefix=PREFIX,
    neighbor_values=COORD_NEIGHBORS,
    metrics=DISTANCE_METRICS,
    weights_values=WEIGHT_OPTIONS,
    cluster_values=CLUSTER_VALUES,
    output_csv=COORD_RESULTS_PATH,
)
print("Experimento de coordenadas finalizado.")


Experimento de coordenadas finalizado.


### 11.1. Búsqueda global con validación


In [36]:
coordinate_tuning = pd.read_csv(COORD_TUNING_PATH)
coordinate_tuning = coordinate_tuning.sort_values(
    ["kind", "rmse_2d_m", "n_neighbors"], ascending=[True, True, True]
).reset_index(drop=True)

print("La selección se realiza con RMSE radial 2D de validación.")
display(coordinate_tuning)


La selección se realiza con RMSE radial 2D de validación.


,kind,n_neighbors,distance,weights,rmse_2d_m,mean_error_m,median_error_m,p75_error_m,p95_error_m,n
0,central,11,manhattan,distance,3.599007,2.974044,2.594825,3.932428,6.759430,1012
1,central,15,manhattan,distance,3.626166,3.012204,2.584651,3.910647,6.964151,1012
2,central,11,manhattan,uniform,3.634040,3.005215,2.607738,3.963459,6.807526,1012
3,central,7,manhattan,distance,3.643402,2.991991,2.537201,3.918089,6.949472,1012
4,central,15,manhattan,uniform,3.666295,3.048265,2.637499,3.980806,6.995585,1012
...,...,...,...,...,...,...,...,...,...,...
59,distributed,3,euclidean,uniform,4.533878,3.623207,2.972257,4.677791,9.044712,1012
60,distributed,1,manhattan,uniform,4.651053,3.642984,2.977000,5.101620,9.157627,1012
61,distributed,1,manhattan,distance,4.651053,3.642984,2.977000,5.101620,9.157627,1012
62,distributed,1,euclidean,uniform,5.387840,4.204349,3.373253,5.661044,10.672091,1012


### 11.2. Selección del número de clústeres


In [37]:
coordinate_cluster_tuning = pd.read_csv(COORD_CLUSTER_TUNING_PATH)
coordinate_cluster_tuning = coordinate_cluster_tuning.sort_values(
    ["modality", "rmse_2d_m", "n_clusters"], ascending=[True, True, True]
).reset_index(drop=True)

display(
    coordinate_cluster_tuning[
        [
            "modality",
            "n_clusters",
            "rmse_2d_m",
            "mean_error_m",
            "coverage",
            "gate_accuracy_diagnostic",
        ]
    ]
)


,modality,n_clusters,rmse_2d_m,mean_error_m,coverage,gate_accuracy_diagnostic
0,centralized_by_predicted_cluster,8,3.336403,2.725772,1.0,0.806324
1,centralized_by_predicted_cluster,6,3.391630,2.811219,1.0,0.809289
2,centralized_by_predicted_cluster,7,3.406162,2.814482,1.0,0.813241
3,centralized_by_predicted_cluster,3,3.407708,2.889940,1.0,0.928854
4,centralized_by_predicted_cluster,10,3.456730,2.842131,1.0,0.728261
5,centralized_by_predicted_cluster,9,3.471996,2.874797,1.0,0.736166
6,centralized_by_predicted_cluster,4,3.494484,2.937892,1.0,0.911067
7,centralized_by_predicted_cluster,5,3.494751,2.931173,1.0,0.857708
8,centralized_by_predicted_cluster,2,3.556435,2.977589,1.0,0.975296
9,federated_by_predicted_cluster,8,3.336403,2.725772,1.0,0.806324


### 11.3. Resultados de validación y test


In [38]:
print("=== Coordenadas: validación ===")
display(compact_results(coordinate_results, split="val"))

print("=== Coordenadas: test final ===")
display(compact_results(coordinate_results, split="test"))


=== Coordenadas: validación ===


,model,modality,n_clusters,rmse_2d_m,mean_error_m,median_error_m,p75_error_m,p95_error_m,coverage,gate_accuracy_diagnostic
0,KNN,centralized_by_predicted_cluster,8.0,3.336403,2.725772,2.217245,3.505717,6.831134,1.0,0.806324
1,KNN,federated_by_predicted_cluster,8.0,3.336403,2.725772,2.217245,3.505717,6.831134,1.0,0.806324
2,KNN,centralized_global,NaN,3.599007,2.974044,2.594825,3.932428,6.759430,NaN,NaN
3,KNN,federated_global_exact_distributed_knn,NaN,3.599007,2.974044,2.594825,3.932428,6.759430,NaN,NaN


=== Coordenadas: test final ===


,model,modality,n_clusters,rmse_2d_m,mean_error_m,median_error_m,p75_error_m,p95_error_m,coverage,gate_accuracy_diagnostic
0,KNN,centralized_global,NaN,3.509080,2.960108,2.566513,4.033164,6.500569,NaN,NaN
1,KNN,federated_global_exact_distributed_knn,NaN,3.509080,2.960108,2.566513,4.033164,6.500569,NaN,NaN
2,KNN,centralized_by_predicted_cluster,8.0,3.608767,2.934945,2.420708,3.859720,6.952323,1.0,0.743363
3,KNN,federated_by_predicted_cluster,8.0,3.608771,2.934953,2.420708,3.859720,6.952323,1.0,0.743363


### 11.4. Coherencia del KNN distribuido exacto


In [39]:
coord_test = coordinate_results[coordinate_results["split"] == "test"].set_index("modality")

central_rmse = float(coord_test.loc["centralized_global", "rmse_2d_m"])
distributed_rmse = float(
    coord_test.loc["federated_global_exact_distributed_knn", "rmse_2d_m"]
)
global_difference = abs(central_rmse - distributed_rmse)
print(f"Diferencia RMSE global central-distribuido: {global_difference:.12f} m")

central_cluster_k = int(coord_test.loc["centralized_by_predicted_cluster", "n_clusters"])
federated_cluster_k = int(coord_test.loc["federated_by_predicted_cluster", "n_clusters"])
print("K central por clúster:", central_cluster_k)
print("K distribuido por clúster:", federated_cluster_k)

if central_cluster_k == federated_cluster_k:
    cluster_difference = abs(
        float(coord_test.loc["centralized_by_predicted_cluster", "rmse_2d_m"])
        - float(coord_test.loc["federated_by_predicted_cluster", "rmse_2d_m"])
    )
    print(f"Diferencia RMSE por clúster central-distribuido: {cluster_difference:.12f} m")

if global_difference > 1e-6:
    print("Aviso: revisa empates exactos de distancia; no debería haber diferencias materiales.")


Diferencia RMSE global central-distribuido: 0.000000000000 m
K central por clúster: 8
K distribuido por clúster: 8
Diferencia RMSE por clúster central-distribuido: 0.000003848401 m


## 12. Hiperparámetros seleccionados


In [40]:
selected_files = [COORD_SELECTED_PATH]
if FLOOR_TASK:
    selected_files.append(FLOOR_SELECTED_PATH)

selected_hyperparameters = {}
for path in selected_files:
    with path.open("r", encoding="utf-8") as handle:
        selected_hyperparameters[path.name] = json.load(handle)

print(json.dumps(selected_hyperparameters, indent=2, ensure_ascii=False))


{
  "tuji1_official_knn_coordinates_corrected_selected_hyperparameters.json": {
    "model": "KNN",
    "task": "coordinates",
    "selection_split": "validation",
    "selection_metric": "rmse_2d_m",
    "n_neighbors": 11,
    "distance": "manhattan",
    "weights": "distance",
    "cluster_protocol": "K is selected independently by modality on validation."
  }
}


## 13. Resumen final


In [41]:
print("=== Resumen final de coordenadas ===")
display(compact_results(coordinate_results, split="test"))

if FLOOR_TASK:
    print("=== Resumen final de planta ===")
    display(compact_floor_results(floor_results, split="test"))


=== Resumen final de coordenadas ===


,model,modality,n_clusters,rmse_2d_m,mean_error_m,median_error_m,p75_error_m,p95_error_m,coverage,gate_accuracy_diagnostic
0,KNN,centralized_global,NaN,3.509080,2.960108,2.566513,4.033164,6.500569,NaN,NaN
1,KNN,federated_global_exact_distributed_knn,NaN,3.509080,2.960108,2.566513,4.033164,6.500569,NaN,NaN
2,KNN,centralized_by_predicted_cluster,8.0,3.608767,2.934945,2.420708,3.859720,6.952323,1.0,0.743363
3,KNN,federated_by_predicted_cluster,8.0,3.608771,2.934953,2.420708,3.859720,6.952323,1.0,0.743363


## 14. Ficheros generados


In [42]:
generated_files = [
    COORD_RESULTS_PATH,
    COORD_TUNING_PATH,
    COORD_CLUSTER_TUNING_PATH,
    COORD_SELECTED_PATH,
]
if FLOOR_TASK:
    generated_files.extend(
        [
            FLOOR_RESULTS_PATH,
            FLOOR_TUNING_PATH,
            FLOOR_CLUSTER_TUNING_PATH,
            FLOOR_SELECTED_PATH,
        ]
    )

display(
    pd.DataFrame(
        [
            {
                "file": str(path.relative_to(ROOT)),
                "exists": path.exists(),
                "size_bytes": path.stat().st_size if path.exists() else 0,
            }
            for path in generated_files
        ]
    )
)


,file,exists,size_bytes
0,results/TUJI1/tuji1_official_knn_coordinates_c...,True,1543
1,results/TUJI1/tuji1_official_knn_coordinates_c...,True,8238
2,results/TUJI1/tuji1_official_knn_coordinates_c...,True,3469
3,results/TUJI1/tuji1_official_knn_coordinates_c...,True,268
